In [17]:
import pandas as pd
import wikipedia
import time

ModuleNotFoundError: No module named 'wikipedia'

In [6]:
data = pd.read_csv("billionaire_individual_perfect.csv")

data.dtypes

df = pd.DataFrame(data)

df

,name,age,net_worth_million_usd,industries,country,self_made,gender
0,Elon Musk,54,800629.06,Technology,United States,True,M
1,Larry Page,53,260898.20,Technology,United States,True,M
2,Jeff Bezos,62,248778.91,Technology,United States,True,M
3,Sergey Brin,52,240743.24,Technology,United States,True,M
4,Mark Zuckerberg,41,216089.31,Technology,United States,True,M
...,...,...,...,...,...,...,...
3369,Robert Johnson,80,1000.00,Media & Entertainment,United States,True,M
3370,Shayne Coplan,27,1000.00,Technology,United States,True,M
3371,Stephen Ketchum,64,1000.00,Finance & Investments,United States,True,M
3372,Thor Bjorgolfsson,59,1000.00,Diversified,Iceland,True,M


In [7]:
df.shape

(3374, 7)

In [10]:
df_self_made = df[df["self_made"] == True]

print(df_self_made.shape)   # rows, columns
print(df_self_made["self_made"].unique())  # should show only True

number_of_self_made_billionaires = df_self_made.shape[0]

print("\n\n\n")
print(f"number of self made billionaires is {number_of_self_made_billionaires} and number of not self made is {df.shape[0] - number_of_self_made_billionaires}")
print(f"percentage of self made billionaires is {number_of_self_made_billionaires / df.shape[0] * 100:.2f}%")
print(f"percentage of not self made billionaires is {(df.shape[0] - number_of_self_made_billionaires) / df.shape[0] * 100:.2f}%")

(2245, 7)
[ True]




number of self made billionaires is 2245 and number of not self made is 1129
percentage of self made billionaires is 66.54%
percentage of not self made billionaires is 33.46%


In [ ]:
df_self_made = df.drop(columns=["self_made"])

df_self_made.sample(n=5)


,name,age,net_worth_million_usd,industries,country,gender
3250,Regina Helena S. Velloso,65,1082.53,Diversified,Brazil,M
1194,William Wrigley Jr,62,3567.95,Food & Beverage,United States,M
2174,Diona Teh Li Shian,65,1896.64,Finance & Investments,Malaysia,F
724,Huang Li,62,5660.11,Technology,China,M
2758,Adam Norwitt,56,1386.60,Manufacturing,United States,M


In [ ]:
# Initialize columns
df["teen_millionaire"] = ""
df["early_success"] = ""

# -----------------------
# 2. Setup LLM
# -----------------------
from openai import OpenAI

# OpenRouter setup
client = OpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1"
)

def classify_bio(bio):
    if not bio or len(bio.strip()) == 0:
        return "unknown", "unknown"

    prompt = f"""
Given this biography:

{bio}

Answer strictly:

1. Did the person become a millionaire at age ≤ 19?
2. Did the person achieve major financial success before age 25?

Rules:
- Only say "yes" if explicitly stated
- Do NOT assume or infer
- If unclear, say "unknown"

Output format:
teen_millionaire: yes/no/unknown
early_success: yes/no/unknown
"""

    try:
        response = client.chat.completions.create(
            model="qwen/qwen3-next-80b-a3b-instruct:free",
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        text = response.choices[0].message.content.lower()

        teen = "unknown"
        early = "unknown"

        if "teen_millionaire: yes" in text:
            teen = "yes"
        elif "teen_millionaire: no" in text:
            teen = "no"

        if "early_success: yes" in text:
            early = "yes"
        elif "early_success: no" in text:
            early = "no"

        return teen, early

    except Exception as e:
        return "unknown", "unknown"

# -----------------------
# 3. Wikipedia fetch
# -----------------------
def get_bio(name):
    try:
        return wikipedia.summary(name, sentences=15)
    except:
        return ""

# -----------------------
# 4. Batch processing
# -----------------------
batch_size = 300

for start in range(0, len(df), batch_size):
    end = start + batch_size
    print(f"Processing rows {start} to {end}")

    for i in range(start, min(end, len(df))):
        name = df.loc[i, "name"]

        bio = get_bio(name)

        
        if any(k in bio.lower() for k in ["million", "founded", "age", "young"]):
            teen, early = classify_bio(bio)
        else:
            teen, early = "unknown", "unknown"

        df.loc[i, "teen_millionaire"] = teen
        df.loc[i, "early_success"] = early

        time.sleep(0.5)  # avoid rate limits

# -----------------------
# 5. Save results
# -----------------------
df.to_csv("df_of_self_made_with_teen_millionaire.csv", index=False)